# A check can read only declared context fields.

Run the cells in order. The matching [script](10_context_in_the_check.py) has the same declarations, calls and assertions.

See the [guide](../../docs/how-to/roles-and-object-rules.md).

## Declarations

Declare the domain, role requirements and typed operation before asking the machine.

In [1]:
"""A check can read only declared context fields."""

from __future__ import annotations

import asyncio

from pydantic import Field

from aoa.action_machine.auth import ApplicationRole
from aoa.action_machine.context import Context
from aoa.action_machine.context.context_view import ContextView
from aoa.action_machine.context.user_info import UserInfo
from aoa.action_machine.domain.base_domain import BaseDomain
from aoa.action_machine.intents.access_control import Allowed, Verdict
from aoa.action_machine.intents.access_decide import access_decide
from aoa.action_machine.intents.aspects import summary_aspect
from aoa.action_machine.intents.check_roles import check_roles
from aoa.action_machine.intents.context_requires import context_requires
from aoa.action_machine.intents.meta import meta
from aoa.action_machine.model import BaseAction, BaseParams, BaseResult
from aoa.action_machine.model.base_state import BaseState
from aoa.action_machine.resources import Connections
from aoa.action_machine.runtime.action_product_machine import ActionProductMachine
from aoa.action_machine.runtime.tools_box import ToolsBox

# %% Setup


class StoreDomain(BaseDomain):
    """Group the order operations."""

    name = "store"
    description = "Order management"


class ManagerRole(ApplicationRole):
    """Permit order management."""

    name = "manager"
    description = "Can manage orders"


class OrderParams(BaseParams):
    """Identify the order to cancel."""

    order_id: str = Field(description="Order identifier")


class OrderResult(BaseResult):
    """Report the cancelled order."""

    order_id: str = Field(description="Cancelled order identifier")


@meta(description="Cancel an order", domain=StoreDomain)
@check_roles(ManagerRole)
class CancelOrderAction(BaseAction[OrderParams, OrderResult]):
    """Cancel an order after checking access."""

    @access_decide("Check whether the order can be cancelled")
    @context_requires("user.user_id")
    async def cancel_access_decide(
        self,
        params: OrderParams,
        box: ToolsBox,
        connections: Connections,
        ctx: ContextView,
    ) -> Verdict:
        """Decide access before cancelling the order."""
        print("declared user:", ctx.get("user.user_id"))
        ctx.get("request.client_ip")
        return Allowed()

    @summary_aspect("Cancel the order")
    async def cancel_summary(self, params: OrderParams, state: BaseState, box: ToolsBox, connections: Connections) -> OrderResult:
        """Return the cancellation result."""
        return OrderResult(order_id=params.order_id)


## Step 1

Run the related calls together. Compare the actual answers or expected declaration errors before continuing.

In [2]:
machine = ActionProductMachine(cache_coordinator=None)
caller = Context(user=UserInfo(user_id="m-1", roles=(ManagerRole,)))


## Step 2

Run the related calls together. Compare the actual answers or expected declaration errors before continuing.

In [3]:
answer = await machine.check_access_decide(caller, CancelOrderAction, OrderParams(order_id="ord-001"))
print("answer:", answer.model_dump(mode="json"))
print("server cause:", type(answer.cause).__name__)


declared user: m-1
answer: {'kind': 'undecided', 'gate': 'ACCESS_DECIDE'}
server cause: ContextAccessError
